# Submit the Serverless Spark YAML

This notebook loads and submits [`serverless-spark-managed-identity.yml`](./serverless-spark-managed-identity.yml). Complete [README.md](./README.md) before enabling submission.

**Official sources:** [job YAML](https://github.com/Azure/azureml-examples/blob/86d53a35d2ecc33ff40c669cc1389b06899e5a4b/cli/jobs/spark/serverless-spark-standalone-managed-identity.yml), [PySpark code](https://github.com/Azure/azureml-examples/blob/86d53a35d2ecc33ff40c669cc1389b06899e5a4b/cli/jobs/spark/src/titanic.py), [Titanic data](https://github.com/Azure/azureml-examples/blob/86d53a35d2ecc33ff40c669cc1389b06899e5a4b/cli/jobs/spark/data/titanic.csv), and [Spark guide](https://github.com/Azure/azureml-examples/blob/86d53a35d2ecc33ff40c669cc1389b06899e5a4b/cli/jobs/spark/README.md).

In [1]:
from pathlib import Path
import os

from azure.ai.ml import MLClient, load_job
from azure.core.exceptions import HttpResponseError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "AGENDA.md").is_file() and (candidate / ".env.example").is_file():
        WORKSHOP_ROOT = candidate
        break
    if (candidate / "workshop/AGENDA.md").is_file():
        WORKSHOP_ROOT = candidate / "workshop"
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

load_dotenv(WORKSHOP_ROOT / ".env", override=True)
credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(
    credential,
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)
job = load_job(
    WORKSHOP_ROOT
    / "notebooks/06_spark_jobs_pipelines/serverless-spark-managed-identity.yml"
)
RUN_JOB = os.getenv("RUN_SERVERLESS_SPARK_JOB", "false").lower() in {
    "1", "true", "yes"
}
print(f"Workspace: {ml_client.workspace_name}")
print(f"Job type: {job.type}")
print(f"Spark runtime: {job.resources.runtime_version}")
print(f"Instance type: {job.resources.instance_type}")

Workspace: mlwpubliccc03
Job type: spark
Spark runtime: 3.4
Instance type: Standard_E4S_V3


In [ ]:
if not RUN_JOB:
    print("Submission disabled. Set RUN_SERVERLESS_SPARK_JOB=true in workshop/.env.")
else:
    try:
        submitted = ml_client.jobs.create_or_update(job)
    except HttpResponseError as exc:
        if exc.error_code == "AuthorizationFailure":
            raise RuntimeError(
                "The YAML contains local code/data that must be uploaded to private "
                "workspace Storage. Run this notebook on aml-instance-single-cc01 "
                "after completing README.md Spark setup."
            ) from exc
        raise
    print(f"Submitted: {submitted.name}")
    print(f"Studio: {submitted.studio_url}")
    ml_client.jobs.stream(submitted.name)
    final = ml_client.jobs.get(submitted.name)
    if final.status != "Completed":
        raise RuntimeError(f"Spark job ended with status {final.status}")
    print(f"Final status: {final.status}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
AzureCliCredential.get_token_info fail

Submitted: crimson_hair_9v48qjqn89
Studio: https://ml.azure.com/runs/crimson_hair_9v48qjqn89?wsid=/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25/resourcegroups/rg-aml-ws-public-cc-03/workspaces/mlwpubliccc03&tid=450bb9bd-faf4-4a7f-9fc1-f621f35fcea8
RunId: crimson_hair_9v48qjqn89
Web View: https://ml.azure.com/runs/crimson_hair_9v48qjqn89?wsid=/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25/resourcegroups/rg-aml-ws-public-cc-03/workspaces/mlwpubliccc03

Execution Summary
RunId: crimson_hair_9v48qjqn89
Web View: https://ml.azure.com/runs/crimson_hair_9v48qjqn89?wsid=/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25/resourcegroups/rg-aml-ws-public-cc-03/workspaces/mlwpubliccc03


JobException: Exception : 
 {
    "error": {
        "code": "UserError",
        "message": "The long operation status is Failed",
        "message_parameters": {},
        "details": []
    },
    "environment": "canadacentral",
    "location": "canadacentral",
    "time": "2026-09-30T07:23:27.424801Z",
    "component_name": "commonjobprovider"
} 

: 

## Expected Result

The YAML submits a managed-identity serverless Spark 3.4 job and writes wrangled Titanic CSV output to `workspaceblobstore`. Reset `RUN_SERVERLESS_SPARK_JOB=false` afterward.